In [0]:
%run ./secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_4", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_4", 0o600)

In [0]:
# %python
# import subprocess
# import os

# source = os.path.join(os.environ['DATASET_PATH'], os.environ['DATASET_NAME'])
# tar_path = source + ".tar.gz"

# result = subprocess.run(
#     f"cd {source} && tar czf {tar_path} --exclude='.git' --exclude='.cache' .",
#     shell=True
# )

# if result.returncode == 1:
#     print("Warning: some files changed during archiving, but the archive was created.")
# elif result.returncode != 0:
#     raise Exception(f"tar failed with exit code {result.returncode}")
# else:
#     print("Archive created successfully.")

In [0]:
# %sh
# ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << EOF
# mkdir -p \$HOME/Isaac-GR00T/databricks_datasets/$DATASET_NAME/
# curl -sL -o /tmp/dataset.tar.gz \
#   -H "Authorization: Bearer $DATABRICKS_TOKEN" \
#   "$DATABRICKS_HOST/api/2.0/fs/files$DATASET_PATH/$DATASET_NAME.tar.gz"
# tar xzf /tmp/dataset.tar.gz -C \$HOME/Isaac-GR00T/databricks_datasets/$DATASET_NAME/
# ls -la \$HOME/Isaac-GR00T/databricks_datasets/$DATASET_NAME/
# rm /tmp/dataset.tar.gz
# EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << EOF
# cd \$HOME/Isaac-GR00T/databricks_datasets/$DATASET_NAME/meta/
cd \$HOME/lerobot_datasets/meta/
# curl -sL -o \$HOME/Isaac-GR00T/databricks_datasets/$DATASET_NAME/meta/$MODALITY_JSON \
#   -H "Authorization: Bearer $DATABRICKS_TOKEN" \
#   "$DATABRICKS_HOST/api/2.0/fs/files$MODALITY_FILES_PATH/$MODALITY_JSON"
 curl -sL -o \$HOME/lerobot_datasets/meta/$MODALITY_JSON \
   -H "Authorization: Bearer $DATABRICKS_TOKEN" \
   "$DATABRICKS_HOST/api/2.0/fs/files$MODALITY_FILES_PATH/$MODALITY_JSON"
ls -la \$HOME/lerobot_datasets/meta/
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << EOF
# cd \$HOME/Isaac-GR00T/databricks_datasets/$DATASET_NAME/meta/
cd \$HOME/lerobot_datasets/meta/
# curl -sL -o \$HOME/Isaac-GR00T/databricks_datasets/$DATASET_NAME/meta/$MODALITY_PY \
#   -H "Authorization: Bearer $DATABRICKS_TOKEN" \
#   "$DATABRICKS_HOST/api/2.0/fs/files$MODALITY_FILES_PATH/$MODALITY_PY"
 curl -sL -o \$HOME/lerobot_datasets/meta/$MODALITY_PY \
   -H "Authorization: Bearer $DATABRICKS_TOKEN" \
   "$DATABRICKS_HOST/api/2.0/fs/files$MODALITY_FILES_PATH/$MODALITY_PY"
ls -la \$HOME/lerobot_datasets/meta/
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << EOF
export PATH="\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv
export WANDB_API_KEY=$WANDB_API_KEY
uv run wandb login
cd \$HOME/Isaac-GR00T
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << EOF
cd \$HOME/Isaac-GR00T
rm -f \$HOME/TRAINING_DONE \$HOME/TRAINING_FAILED
tmux kill-session -t finetune 2>/dev/null || true
tmux new-session -d -s finetune
tmux send-keys -t finetune 'export WANDB_API_KEY=$WANDB_API_KEY' C-m
tmux send-keys -t finetune 'export WANDB_NAME=gr00t-n1d6-so100' C-m
tmux send-keys -t finetune 'export PATH=\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin' C-m
tmux send-keys -t finetune 'cd \$HOME/Isaac-GR00T && CUDA_VISIBLE_DEVICES=0 uv run python gr00t/experiment/launch_finetune.py --base_model_path nvidia/GR00T-N1.6-3B --dataset_path ~/lerobot_datasets/ --modality_config_path ~/lerobot_datasets/meta/$MODALITY_PY --embodiment_tag NEW_EMBODIMENT --num_gpus 1 --output_dir ./finetuned_models/$DATASET_NAME/ --save_steps $SAVE_STEPS --save_total_limit 5 --max_steps $MAX_STEPS --use-wandb --warmup_ratio 0.05 --weight_decay 1e-5 --learning_rate 1e-4 --global_batch_size 128 --color_jitter_params brightness 0.3 contrast 0.4 saturation 0.5 hue 0.08 --dataloader_num_workers 3 2>&1 | tee \$HOME/finetune.log && touch \$HOME/TRAINING_DONE || touch \$HOME/TRAINING_FAILED' C-m
tmux send-keys -t finetune 'exit' C-m
EOF

In [0]:
%sh
echo "Waiting for training to complete..."
while true; do
  echo "$(date): Checking status..."
  
  OUTPUT=$(echo '
    echo "=== TMUX SESSIONS ==="
    tmux list-sessions 2>&1 || echo "NO_SESSIONS"
    echo "=== CHECKING FINETUNE ==="
    if tmux has-session -t finetune 2>/dev/null; then
      echo "STATUS_RUNNING"
    else
      echo "STATUS_DONE"
    fi
  ' | ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP 2>&1)
  
  EXIT_CODE=$?
  
  echo "--- SSH Exit Code: $EXIT_CODE ---"
  echo "--- Full Output ---"
  echo "$OUTPUT"
  echo "-------------------"
  
  if [ $EXIT_CODE -ne 0 ]; then
    echo "WARNING: SSH command failed!"
  fi
  
  if echo "$OUTPUT" | grep -q "STATUS_DONE"; then
    echo "Training complete!"
    break
  fi
  
  sleep 60
done

echo "Cleaning up finetuned model files..."

ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << EOF
cd \$HOME/Isaac-GR00T/finetuned_models/$DATASET_NAME/
echo "== Before cleanup =="
du -sh .
# Delete all checkpoints except keep root level model files
# rm -rf checkpoint-*/
# Delete optimizer states, scheduler, rng
# find . -name "optimizer*" -exec rm {} \;
# find . -name "scheduler*" -exec rm {} \;
# find . -name "rng_state*" -exec rm {} \;
# echo "== After cleanup =="
du -sh .
ls -lh
EOF

In [0]:
# %sh
# # Sending the finetuned model folder into 4GB chunks because the PUT API endpoint has a limit of ~5GB per send
# ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP<< EOF
# cd \$HOME/Isaac-GR00T/finetuned_models/
# tar cf - $DATASET_NAME/ | split -b 4G --filter='
#   name=\$(basename \$FILE)
#   echo "Uploading \$name..."
#   curl -s -X PUT \
#     -H "Authorization: Bearer $DATABRICKS_TOKEN" \
#     -H "Content-Type: application/octet-stream" \
#     -T - \
#     "$DATABRICKS_HOST/api/2.0/fs/files/Volumes/workspace/default/trained_models/$DATASET_NAME/\$name"
#   echo "\$name done"
# ' - /tmp/model_chunk_
# echo "All chunks uploaded."
# EOF

In [0]:
%sh
scp -r -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no \
  ubuntu@$BREV_IP:/home/ubuntu/Isaac-GR00T/finetuned_models/ \
  /Volumes/workspace/default/trained_models/

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
sudo shutdown -h now
EOF


In [0]:
# %python
# import subprocess
# import os

# dataset = os.environ['DATASET_NAME']
# chunks_path = f"/Volumes/workspace/default/trained_models/{dataset}"
# output = f"/Volumes/workspace/default/trained_models/{dataset}.tar"

# # Reassemble chunks into single tar
# subprocess.run(f"cat {chunks_path}/model_chunk_* > {output}", shell=True, check=True)
# print("Reassembled.")

# # Extract
# subprocess.run(f"tar xf {output} -C /Volumes/workspace/default/trained_models/", shell=True, check=True)
# print("Extracted.")

# # Cleanup chunks and tar
# subprocess.run(f"rm {chunks_path}/model_chunk_* {output}", shell=True, check=True)
# print("Cleaned up.")